# 1. ProteinMPNN: design sequences for a fixed backbone

- **Input:** The ubiquitin backbone (1UBQ, chain A; 76 residues).
- **Model information:** Backbone geometry and learned relationships between protein sequences and structures.
- **Design objective:** Generate alternative sequences compatible with the same backbone.

Use ProteinMPNN to sample three sequences, compare them with ubiquitin, and predict the structure of a selected design.


## Setup
Select **Runtime → Change runtime type → T4 GPU**, then run the two cells below to load the tools and models.


In [ ]:
!git -C BME305-protein-design-tools pull -q 2>/dev/null || git clone -q https://github.com/RomeroLab/BME305-protein-design-tools.git
%pip -q install --upgrade ./BME305-protein-design-tools
from protein_design_tools import load_pdb, load_ESMFold, load_ESM2, load_MPNN, load_RFdiffusion, predict_structure, MPNNdesign, ESM2design, RFdiffusion, alignment, view_structure, inspect_prediction, save_results


In [ ]:
load_MPNN()
load_ESMFold();


## 1. Start with a structure
Rotate ubiquitin and find its helices, sheets, and loops. This is the structural constraint for sequence design.


In [ ]:
native_seq, reference_pdb = load_pdb('1UBQ', chain='A')
view_structure(reference_pdb)


## 2. Design sequences
ProteinMPNN samples amino acids conditioned on the backbone. Higher temperature usually produces more diverse sequences. We do not require recovery of the natural sequence.


In [ ]:
designs = MPNNdesign(reference_pdb, num_sequences=3, temperature=0.1, seed=7)


## 3. Compare sequences
Compare each design with ubiquitin and identify positions conserved across the designs. `|` marks a match to ubiquitin; `.` marks a substitution.


In [ ]:
alignment({'ubiquitin':native_seq, **designs})


## 4. Predict and compare structures
Choose a design and predict its structure with ESMFold. The predictor uses the sequence alone, without the reference coordinates. RMSD measures agreement with the reference backbone; pLDDT describes confidence in the predicted structure. Consider both when interpreting the comparison.


In [ ]:
prediction_pdb = predict_structure(designs['design_1'], num_recycles=1)
metrics = inspect_prediction(reference_pdb, prediction_pdb)


## Discuss
- Where does the prediction agree with the backbone, and where does it differ? Check confidence at the mismatches.
- Would a matching fold establish that the design retains ubiquitin's biological function? What else would you test?

[ProteinMPNN](https://github.com/dauparas/ProteinMPNN) · [1UBQ](https://www.rcsb.org/structure/1UBQ) · [ESMFold](https://github.com/facebookresearch/esm)


In [ ]:
bundle = save_results(designs, reference_pdb, prediction_pdb)
from google.colab import files
files.download(str(bundle))


The same tools accept your own sequences and structures. See the [function reference](https://github.com/RomeroLab/BME305-protein-design-tools#function-reference) for use in later assignments.
